# Ep. 01 — The AI Engineering map

Course 1 — AI Engineering Fundamentals · Linear [CM-41](https://linear.app/caiomedeiros/issue/CM-41)


## Learning objectives

By the end of this notebook you will be able to:
1. Distinguish **AI Engineer**, **Prompt Engineer**, and **ML Engineer**.
2. Draw the channel map: **data → model → system → product → operations**.
3. Decide which layer a problem belongs to before writing a prompt.


## Hook — three job titles, one confusion

People mix up *prompt engineering*, *ML engineering*, and *AI engineering*. That confusion wastes time and ships fragile demos. This episode fixes the vocabulary, then gives you a practical map.


## Precise definition

**AI Engineering** is the discipline of turning model capability into **reliable product behavior**: contracts, retrieval, tools, guardrails, evaluation, cost/latency, and operations — not only clever prompts.

| Role | Primary focus | Typical artifact |
|------|---------------|------------------|
| Prompt Engineer | Wording & few-shots | Prompt templates |
| ML Engineer | Training / fine-tuning / serving models | Models & training pipelines |
| AI Engineer | System around the model | End-to-end AI features in production |


## Map: data → model → system → product → operations

Every AI feature crosses these layers. Skipping a layer is how demos die in production.

Run the cell below to print the map and score a fictional ticket.


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

# Optional: set OPENAI_API_KEY to call a real OpenAI-compatible API.
# Without a key, every demo below still runs offline with a tiny mock LLM.
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
API_BASE = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")


def chat(messages: list[dict[str, str]], *, temperature: float = 0.2, max_tokens: int = 300) -> str:
    """Vanilla chat completion: direct HTTP to an OpenAI-compatible API, or offline mock."""
    if not API_KEY:
        return _mock_chat(messages)
    try:
        import urllib.request

        payload = {
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        req = urllib.request.Request(
            f"{API_BASE}/chat/completions",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return data["choices"][0]["message"]["content"]
    except Exception as exc:  # noqa: BLE001 — educational fallback
        return f"[API error → offline mock] {exc}\n\n" + _mock_chat(messages)


def _mock_chat(messages: list[dict[str, str]]) -> str:
    """Deterministic offline stand-in so the lesson runs without credentials."""
    user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    lower = user.lower()

    if "json" in system.lower() or "json" in lower:
        return '{"ok": true, "summary": "mock structured answer", "confidence": 0.61}'
    if "capital of france" in lower or "capital de frança" in lower or "capital de francia" in lower:
        return "Paris"
    if "2+2" in lower or "2 + 2" in lower:
        return "4"
    if "invent" in lower or "invente" in lower or "inventa" in lower:
        return "The 1847 Treaty of New Avalon was signed by Admiral Kestrel."
    if "email" in lower or "ssn" in lower or "cpf" in lower:
        return "Sure — contact jane.doe@example.com / SSN 123-45-6789."
    # Default: echo a short, slightly overconfident reply
    snippet = user.strip().replace("\n", " ")[:120]
    return f"Based on my training, the answer is clearly related to: {snippet}…"


print("Mode:", "LIVE API" if API_KEY else "OFFLINE MOCK (set OPENAI_API_KEY for live calls)")
print("Model:", MODEL if API_KEY else "mock-llm")


In [ ]:
LAYERS = [
    ("data", "sources, cleaning, labels, permissions"),
    ("model", "provider choice, prompts, decoding, fine-tunes"),
    ("system", "retrieval, tools, memory, guardrails, orchestration"),
    ("product", "UX contracts, latency SLO, fallbacks, pricing"),
    ("operations", "eval, monitoring, cost, incident response"),
]

ROLE_FOCUS = {
    "prompt_engineer": {"model"},
    "ml_engineer": {"data", "model"},
    "ai_engineer": {"data", "model", "system", "product", "operations"},
}

def print_map() -> None:
    print("AI Engineering map")
    print("=" * 40)
    for name, blurb in LAYERS:
        print(f"→ {name:12} {blurb}")

def score_ticket(description: str, suspected_layer: str) -> dict[str, Any]:
    keywords = {
        "data": ["dataset", "label", "pii", "schema", "csv", "permission"],
        "model": ["prompt", "temperature", "hallucin", "token", "fine-tune"],
        "system": ["retriev", "rag", "tool", "agent", "guardrail", "memory"],
        "product": ["button", "ux", "latency", "sla", "fallback", "pricing"],
        "operations": ["eval", "metric", "monitor", "cost", "incident", "alert"],
    }
    text = description.lower()
    hits = {layer: sum(1 for k in keys if k in text) for layer, keys in keywords.items()}
    guessed = max(hits, key=hits.get)
    return {
        "suspected_layer": suspected_layer,
        "suggested_layer": guessed,
        "keyword_hits": hits,
        "aligned": suspected_layer == guessed,
    }

print_map()
print()
ticket = "Chat answers cite the wrong PDF; retrieval returns unrelated chunks under load."
result = score_ticket(ticket, suspected_layer="system")
print("Ticket:", ticket)
print(json.dumps(result, indent=2))
assert result["suggested_layer"] == "system"
print("\nChecklist: name the broken layer before editing the prompt.")


## Action checklist

Before you open a chat UI, answer: **which layer is broken?** If you cannot name the layer, you are guessing.


## What this channel covers (and what it does not)

- **Covers (Fundamentals):** tokens/context, prompt vs system, basic eval, risks.
- **Later pillars:** deeper systems, production hardening, advanced evals/frameworks.
- **Not here:** training foundation models from scratch.


## Takeaways

1. Prompt skill ≠ AI Engineering.
2. The map forces you to locate the failure before changing words.
3. Next episode: tokens, context windows, and why models “err” without being “dumb”.
